# MammoSight — Comprehensive Analysis Notebook

This notebook covers:
1. **Setup & Data Loading** — predictions CSV, ground truth, model checkpoint
2. **Statistical Significance Testing** — bootstrap paired test, DeLong AUC test, Benjamini-Hochberg correction
3. **Classification Visualizations** — confusion matrices, ROC curves, calibration curves, per-class F1 bars
4. **Segmentation Visualizations** — Dice/IoU distributions, overlay comparisons
5. **Attention & Saliency Visualizations** — GradCAM++, DINO attention, attention rollout (plug-and-play)
6. **Feature Space Analysis** — UMAP/t-SNE of combined features, per-class clustering
7. **Dataset Distribution Analysis** — label distributions, per-source breakdown
8. **Error Analysis** — failure case inspection, confidence vs correctness
9. **Publication-Ready Figure Export**


---
## 0. Imports & Configuration

In [ ]:
# ── Core ────────────────────────────────────────────────────────────────────
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── ML / Stats ──────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import scipy.stats as stats
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize

# ── Dimensionality reduction ─────────────────────────────────────────────────
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not found. Run: pip install umap-learn")
from sklearn.manifold import TSNE

# ── Image ────────────────────────────────────────────────────────────────────
import cv2
from PIL import Image

# ── Visualization toolkit — pytorch-grad-cam ─────────────────────────────────
try:
    from pytorch_grad_cam import (
        GradCAM, GradCAMPlusPlus, ScoreCAM, EigenCAM,
        EigenGradCAM, LayerCAM, FullGrad
    )
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    HAS_GRADCAM = True
except ImportError:
    HAS_GRADCAM = False
    print("grad-cam not found. Run: pip install grad-cam")

print("Imports OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ★  EDIT THESE PATHS TO MATCH YOUR SETUP  ★
# ═══════════════════════════════════════════════════════════════════════════════

# Prediction CSVs produced by test.py
MAMMOSIGHT_PRED_CSV   = "../multi_task_supervised_model_final_data/best_model_medimageinsight_final_data_run_ep8/predictions_test.csv"

# Master split CSV
MASTER_CSV            = "mammo-bench_master_split.csv"

# Image root (for saliency visualizations)
IMAGE_ROOT            = "/data/gaurav.bhole/FullDataset/"

# Model checkpoint (for saliency / feature extraction)
CHECKPOINT_PATH       = "/data/gaurav.bhole/medimageinsights_final_data_run/best_model_medimageinsights_final_data_run.pth"

# Add your project root to path so we can import MammoSight modules
PROJECT_ROOT          = "/home/gourav/Mammography_Project/multi_task_supervised_model_final_data"
sys.path.insert(0, PROJECT_ROOT)

# Output directory for figures
FIG_DIR               = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Global aesthetics ────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
PALETTE = sns.color_palette("tab10")

# ── Task definitions ─────────────────────────────────────────────────────────
TASKS = {
    "classification": ["Normal", "Benign", "Malignant"],
    "density":        ["A", "B", "C", "D"],
    "birads":         ["0", "1", "2", "3", "4", "5"],
    "abnormality":    ["Normal", "Mass", "Calc"],
}
TASK_DISPLAY = {
    "classification": "Cancer Severity",
    "density":        "Breast Density",
    "birads":         "BI-RADS",
    "abnormality":    "Abnormality",
}
# Baseline model prediction CSVs (optional — set to None to skip)
# If you have extracted predictions for baselines in the same format, add here
BASELINE_PREDS = {
    # "MedGemma-4B":   "path/to/medgemma4b_preds.csv",
    # "MedGemma-27B":  "path/to/medgemma27b_preds.csv",
    # "MedImageInsight": "path/to/medimageinsight_zeroshot_preds.csv",
    # "BiomedCLIP":    "path/to/biomedclip_preds.csv",
}

---
## 1. Data Loading

In [ ]:
def load_predictions(csv_path):
    """
    Load prediction CSV produced by test.py.
    Returns a dict: task -> {targs, preds, probs, valid}
    """
    df = pd.read_csv(csv_path)
    result = {}
    for task in TASKS:
        valid_mask = df[f"{task}_valid"].values.astype(bool)
        targs  = df[f"{task}_target"].values[valid_mask]
        preds  = df[f"{task}_pred"].values[valid_mask]
        n_cls  = len(TASKS[task])
        prob_cols = [f"{task}_prob_class{i}" for i in range(n_cls)]
        probs  = df[prob_cols].values[valid_mask]
        result[task] = {
            "targs": targs, "preds": preds, "probs": probs,
            "breast_id": df["breast_id"].values[valid_mask],
            "source": df["source_dataset"].values[valid_mask],
        }
        print(f"  {task:<15}: {len(targs):>5} valid samples")
    return df, result

print("Loading MammoSight predictions...")
mammo_df, mammo_preds = load_predictions(MAMMOSIGHT_PRED_CSV)

# Load baselines if available
baseline_preds_all = {}
for name, path in BASELINE_PREDS.items():
    if path and os.path.exists(path):
        print(f"\nLoading {name} predictions...")
        _, bp = load_predictions(path)
        baseline_preds_all[name] = bp

print("\nData loading complete.")

---
## 2. Statistical Significance Testing

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  2.1  Bootstrap Paired Test for Macro F1
# ────────────────────────────────────────────────────────────────────────────

def bootstrap_f1_test(y_true, y_pred_a, y_pred_b,
                      n_bootstrap=10_000, seed=42, alternative='two-sided'):
    """
    Paired bootstrap hypothesis test for difference in macro F1.

    H0: F1(A) == F1(B)
    Returns: observed_delta, CI_low, CI_high, p_value
    """
    rng = np.random.default_rng(seed)
    n   = len(y_true)

    f1_a_obs = f1_score(y_true, y_pred_a, average='macro', zero_division=0)
    f1_b_obs = f1_score(y_true, y_pred_b, average='macro', zero_division=0)
    observed_delta = f1_a_obs - f1_b_obs

    boot_deltas = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        fa  = f1_score(y_true[idx], y_pred_a[idx], average='macro', zero_division=0)
        fb  = f1_score(y_true[idx], y_pred_b[idx], average='macro', zero_division=0)
        boot_deltas[i] = fa - fb

    ci_low  = np.percentile(boot_deltas, 2.5)
    ci_high = np.percentile(boot_deltas, 97.5)

    if alternative == 'two-sided':
        p_val = np.mean(np.abs(boot_deltas) >= np.abs(observed_delta))
    elif alternative == 'greater':   # A > B
        p_val = np.mean(boot_deltas <= 0)
    else:                            # A < B
        p_val = np.mean(boot_deltas >= 0)

    return {
        'f1_a': f1_a_obs, 'f1_b': f1_b_obs,
        'delta': observed_delta,
        'ci_low': ci_low, 'ci_high': ci_high,
        'p_value': float(p_val)
    }


def bootstrap_metric_ci(y_true, y_pred, metric_fn,
                        n_bootstrap=10_000, seed=42):
    """
    Bootstrap 95% CI for any scalar metric.
    metric_fn(y_true, y_pred) -> scalar
    """
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    obs = metric_fn(y_true, y_pred)
    boot = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        boot[i] = metric_fn(y_true[idx], y_pred[idx])

    return obs, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

print("Bootstrap testing functions defined.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  2.2  DeLong's Test for AUC Comparison
# ────────────────────────────────────────────────────────────────────────────

def delong_auc_test(y_true, prob_a, prob_b):
    """
    DeLong non-parametric test for comparing two AUC values on the same test set.
    Works for binary classification.
    For multiclass: pass in one-vs-rest probabilities per class.

    Reference: DeLong et al. (1988) Biometrics 44(3):837-845
    Returns: auc_a, auc_b, z_statistic, p_value
    """
    def _auc_placement(y, prob):
        pos = prob[y == 1]
        neg = prob[y == 0]
        m, n = len(pos), len(neg)
        # Placement values (structural components)
        V10 = np.array([np.mean(p > neg) + 0.5 * np.mean(p == neg) for p in pos])
        V01 = np.array([np.mean(n_ < pos) + 0.5 * np.mean(n_ == pos) for n_ in neg])
        auc_val = V10.mean()
        S10 = np.var(V10, ddof=1) / m
        S01 = np.var(V01, ddof=1) / n
        return auc_val, V10, V01, S10, S01, m, n

    auc_a, V10_a, V01_a, S10_a, S01_a, m, n = _auc_placement(y_true, prob_a)
    auc_b, V10_b, V01_b, S10_b, S01_b, _, _ = _auc_placement(y_true, prob_b)

    # Covariance between the two AUC estimators
    S10_ab = np.cov(V10_a, V10_b, ddof=1)[0, 1] / m
    S01_ab = np.cov(V01_a, V01_b, ddof=1)[0, 1] / n

    var_diff = S10_a + S01_a + S10_b + S01_b - 2 * S10_ab - 2 * S01_ab
    if var_diff <= 0:
        return auc_a, auc_b, 0.0, 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))  # two-tailed
    return auc_a, auc_b, z, p


def multiclass_delong(y_true, probs_a, probs_b, classes):
    """
    One-vs-rest DeLong test for multiclass AUC.
    Returns dict of per-class results + macro average.
    """
    y_bin = label_binarize(y_true, classes=list(range(len(classes))))
    results = {}
    for k, cls in enumerate(classes):
        if y_bin[:, k].sum() < 2:
            continue
        auc_a, auc_b, z, p = delong_auc_test(
            y_bin[:, k], probs_a[:, k], probs_b[:, k]
        )
        results[cls] = {'auc_a': auc_a, 'auc_b': auc_b, 'z': z, 'p': p}
    return results

print("DeLong test functions defined.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  2.3  Benjamini-Hochberg FDR Correction
# ────────────────────────────────────────────────────────────────────────────

def benjamini_hochberg(p_values, alpha=0.05):
    """
    Benjamini-Hochberg procedure for controlling FDR.
    Returns adjusted p-values and rejection mask.
    Reference: Benjamini & Hochberg (1995) JRSS-B 57(1):289-300
    """
    p = np.array(p_values)
    n = len(p)
    order = np.argsort(p)
    ranked_p = p[order]
    thresholds = (np.arange(1, n+1) / n) * alpha
    reject = ranked_p <= thresholds
    # Any p beyond the last rejected one is also rejected (monotone step)
    if reject.any():
        last = np.where(reject)[0][-1]
        reject[:last+1] = True
    adj_p = np.minimum(1.0, ranked_p * n / np.arange(1, n+1))
    # Restore original order
    adj_p_orig  = np.empty(n)
    reject_orig = np.empty(n, dtype=bool)
    adj_p_orig[order]  = adj_p
    reject_orig[order] = reject
    return adj_p_orig, reject_orig

print("BH correction defined.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  2.4  Compute Bootstrap CIs for MammoSight on all tasks
# ────────────────────────────────────────────────────────────────────────────

print("Computing 95% bootstrap CIs for MammoSight (10,000 iterations)...\n")

ci_results = {}
for task in tqdm(TASKS, desc="Tasks"):
    d = mammo_preds[task]
    y_true, y_pred = d['targs'], d['preds']

    f1_obs, f1_lo, f1_hi = bootstrap_metric_ci(
        y_true, y_pred,
        lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0)
    )
    acc_obs, acc_lo, acc_hi = bootstrap_metric_ci(
        y_true, y_pred, accuracy_score
    )
    mcc_obs, mcc_lo, mcc_hi = bootstrap_metric_ci(
        y_true, y_pred, matthews_corrcoef
    )
    bal_obs, bal_lo, bal_hi = bootstrap_metric_ci(
        y_true, y_pred, balanced_accuracy_score
    )
    ci_results[task] = {
        'f1':  (f1_obs,  f1_lo,  f1_hi),
        'acc': (acc_obs, acc_lo, acc_hi),
        'mcc': (mcc_obs, mcc_lo, mcc_hi),
        'bal': (bal_obs, bal_lo, bal_hi),
    }
    print(f"  {TASK_DISPLAY[task]:<20}  "
          f"Macro F1 = {f1_obs:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]  "
          f"MCC = {mcc_obs:.4f} [{mcc_lo:.4f}, {mcc_hi:.4f}]")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  2.5  Pairwise Significance Tests: MammoSight vs Each Baseline
#       (Run this cell only if BASELINE_PREDS is populated)
# ────────────────────────────────────────────────────────────────────────────

if baseline_preds_all:
    all_p_values = []
    all_labels   = []

    comparison_rows = []
    for task in tqdm(TASKS, desc="Comparisons"):
        ms_true = mammo_preds[task]['targs']
        ms_pred = mammo_preds[task]['preds']
        for bname, bpreds in baseline_preds_all.items():
            # Align on common breast_ids
            ms_ids = set(mammo_preds[task]['breast_id'])
            bl_ids = set(bpreds[task]['breast_id'])
            common = ms_ids & bl_ids
            if len(common) < 20:
                continue

            ms_mask = np.isin(mammo_preds[task]['breast_id'], list(common))
            bl_mask = np.isin(bpreds[task]['breast_id'], list(common))
            yt  = mammo_preds[task]['targs'][ms_mask]
            ypa = mammo_preds[task]['preds'][ms_mask]
            ypb = bpreds[task]['preds'][bl_mask]

            res = bootstrap_f1_test(yt, ypa, ypb)
            comparison_rows.append({
                'Task': TASK_DISPLAY[task], 'Baseline': bname,
                'MammoSight F1': res['f1_a'], 'Baseline F1': res['f1_b'],
                'Delta': res['delta'],
                '95% CI': f"[{res['ci_low']:.4f}, {res['ci_high']:.4f}]",
                'p-value (raw)': res['p_value']
            })
            all_p_values.append(res['p_value'])
            all_labels.append(f"{task} vs {bname}")

    # BH correction
    if all_p_values:
        adj_p, rejected = benjamini_hochberg(all_p_values)
        for i, row in enumerate(comparison_rows):
            row['p-value (BH-adj)'] = adj_p[i]
            row['Significant (α=0.05)'] = '✓' if rejected[i] else '✗'

    cmp_df = pd.DataFrame(comparison_rows)
    display(cmp_df.style.format({
        'MammoSight F1': '{:.4f}', 'Baseline F1': '{:.4f}',
        'Delta': '{:+.4f}', 'p-value (raw)': '{:.4f}',
        'p-value (BH-adj)': '{:.4f}'
    }).background_gradient(subset=['Delta'], cmap='RdYlGn'))
    cmp_df.to_csv(FIG_DIR / 'significance_tests.csv', index=False)
    print(f"\nSaved → {FIG_DIR}/significance_tests.csv")
else:
    print("No baseline prediction CSVs provided.")
    print("To use this cell, populate BASELINE_PREDS in the config cell above.")
    print("\nRunning self-contained CI display instead:")
    rows = []
    for task, res in ci_results.items():
        rows.append({
            'Task': TASK_DISPLAY[task],
            'Macro F1': f"{res['f1'][0]:.4f} [{res['f1'][1]:.4f}, {res['f1'][2]:.4f}]",
            'Accuracy': f"{res['acc'][0]:.4f} [{res['acc'][1]:.4f}, {res['acc'][2]:.4f}]",
            'MCC':      f"{res['mcc'][0]:.4f} [{res['mcc'][1]:.4f}, {res['mcc'][2]:.4f}]",
            'Bal. Acc': f"{res['bal'][0]:.4f} [{res['bal'][1]:.4f}, {res['bal'][2]:.4f}]",
        })
    display(pd.DataFrame(rows))

---
## 3. Classification Visualizations

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.1  Confusion Matrices (Normalized) — 2×2 Grid
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, task in tqdm(zip(axes, TASKS), desc="Plotting Confusion Matrices"):
    d = mammo_preds[task]
    cls_names = TASKS[task]
    n_cls = len(cls_names)
    cm = confusion_matrix(d['targs'], d['preds'], labels=list(range(n_cls)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=cls_names, yticklabels=cls_names,
        ax=ax, linewidths=0.5, vmin=0, vmax=1,
        annot_kws={'size': 9}
    )
    # Overlay raw counts
    for i in range(n_cls):
        for j in range(n_cls):
            ax.text(j+0.5, i+0.72, f'({cm[i,j]})',
                    ha='center', va='center', fontsize=7, color='gray')

    f1 = f1_score(d['targs'], d['preds'], average='macro', zero_division=0)
    ax.set_title(f"{TASK_DISPLAY[task]}\nMacro F1 = {f1:.4f}", fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('MammoSight — Normalized Confusion Matrices (row-wise)\nRaw counts in parentheses',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrices.pdf')
plt.show()
print(f"Saved → {FIG_DIR}/confusion_matrices.pdf")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.2  Per-Class F1 Bar Charts with Bootstrap CI
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=False)

for ax, task in tqdm(zip(axes, TASKS), desc="Plotting Per-Class F1 Scores"):
    d = mammo_preds[task]
    cls_names = TASKS[task]
    n_cls = len(cls_names)

    # Per-class F1 with bootstrap CI
    f1_per_class = f1_score(d['targs'], d['preds'],
                            average=None, zero_division=0,
                            labels=list(range(n_cls)))

    # Bootstrap CI per class
    rng = np.random.default_rng(42)
    n = len(d['targs'])
    boot_f1 = np.zeros((10000, n_cls))
    for b in range(10000):
        idx = rng.integers(0, n, n)
        boot_f1[b] = f1_score(d['targs'][idx], d['preds'][idx],
                              average=None, zero_division=0,
                              labels=list(range(n_cls)))
    ci_lo = np.percentile(boot_f1, 2.5, axis=0)
    ci_hi = np.percentile(boot_f1, 97.5, axis=0)
    yerr = np.array([f1_per_class - ci_lo, ci_hi - f1_per_class])

    colors = [PALETTE[i] for i in range(n_cls)]
    bars = ax.bar(cls_names, f1_per_class, color=colors,
                  yerr=yerr, capsize=5, edgecolor='black', linewidth=0.5,
                  error_kw={'elinewidth': 1.5, 'ecolor': 'black'})
    for bar, val in zip(bars, f1_per_class):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    # Sample counts per class
    counts = [np.sum(d['targs'] == k) for k in range(n_cls)]
    ax.set_xticks(range(n_cls))
    ax.set_xticklabels([f"{c}\n(n={cnt})" for c, cnt in zip(cls_names, counts)])
    ax.set_ylim(0, 1.15)
    ax.set_title(TASK_DISPLAY[task], fontweight='bold')
    ax.set_ylabel('F1 Score')
    ax.axhline(y=np.mean(f1_per_class), color='red', linestyle='--',
               linewidth=1.2, label=f'Macro mean={np.mean(f1_per_class):.3f}')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Per-Class F1 Score with 95% Bootstrap CI', y=1.02,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'per_class_f1.pdf')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.3  Multi-Class ROC Curves (One-vs-Rest) with AUC
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, task in tqdm(zip(axes, TASKS), desc="Plotting ROC Curves"):
    d = mammo_preds[task]
    cls_names = TASKS[task]
    n_cls = len(cls_names)
    y_bin = label_binarize(d['targs'], classes=list(range(n_cls)))

    # Macro AUC
    try:
        macro_auc = roc_auc_score(d['targs'], d['probs'],
                                  multi_class='ovr', average='macro')
    except:
        macro_auc = float('nan')

    for k in range(n_cls):
        if y_bin[:, k].sum() < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, k], d['probs'][:, k])
        roc_auc_k = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, color=PALETTE[k],
                label=f"{cls_names[k]} (AUC={roc_auc_k:.3f})")

    ax.plot([0,1],[0,1],'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f"{TASK_DISPLAY[task]}\nMacro AUC = {macro_auc:.4f}",
                 fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('MammoSight — One-vs-Rest ROC Curves', y=1.02,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curves.pdf')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.4  Precision-Recall Curves
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, task in tqdm(zip(axes, TASKS), desc="Plotting Precision-Recall Curves"):
    d = mammo_preds[task]
    cls_names = TASKS[task]
    n_cls = len(cls_names)
    y_bin = label_binarize(d['targs'], classes=list(range(n_cls)))

    for k in range(n_cls):
        if y_bin[:, k].sum() < 2:
            continue
        prec, rec, _ = precision_recall_curve(y_bin[:, k], d['probs'][:, k])
        ap = average_precision_score(y_bin[:, k], d['probs'][:, k])
        baseline = y_bin[:, k].mean()
        ax.plot(rec, prec, linewidth=2, color=PALETTE[k],
                label=f"{cls_names[k]} (AP={ap:.3f})")
        ax.axhline(baseline, color=PALETTE[k], linestyle=':', alpha=0.4)

    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(TASK_DISPLAY[task], fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('MammoSight — Precision-Recall Curves\n(dotted lines = random baseline per class)',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'pr_curves.pdf')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.5  Calibration Curves (Reliability Diagrams)
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, task in tqdm(zip(axes, TASKS), desc="Plotting Calibration Curves"):
    d = mammo_preds[task]
    cls_names = TASKS[task]
    n_cls = len(cls_names)
    y_bin = label_binarize(d['targs'], classes=list(range(n_cls)))

    for k in range(n_cls):
        if y_bin[:, k].sum() < 2:
            continue
        try:
            frac_pos, mean_pred = calibration_curve(
                y_bin[:, k], d['probs'][:, k], n_bins=10, strategy='uniform'
            )
            brier = brier_score_loss(y_bin[:, k], d['probs'][:, k])
            ax.plot(mean_pred, frac_pos, 's-', linewidth=2, markersize=5,
                    color=PALETTE[k],
                    label=f"{cls_names[k]} (Brier={brier:.3f})")
        except:
            pass

    ax.plot([0,1],[0,1],'k--', linewidth=1.5, label='Perfect calibration')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(TASK_DISPLAY[task], fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('MammoSight — Calibration Curves (Reliability Diagrams)\nBrier score in legend',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'calibration_curves.pdf')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  3.6  Model Comparison Radar Chart
#       Works even without baselines — shows MammoSight alone
# ────────────────────────────────────────────────────────────────────────────

# Hard-coded comparison values from your results tables
# Format: {model: {task: macro_f1}}
comparison_f1 = {
    'MammoSight': {
        'classification': 0.7015, 'density': 0.6716,
        'birads': 0.4262, 'abnormality': 0.7215
    },
    'MedGemma-4B': {
        'classification': 0.7130, 'density': 0.6457,
        'birads': 0.4436, 'abnormality': 0.7796
    },
    'MedGemma-27B': {
        'classification': 0.7132, 'density': 0.6428,
        'birads': 0.4438, 'abnormality': 0.7886
    },
    'MedImageInsight': {
        'classification': 0.6761, 'density': 0.6305,
        'birads': 0.4613, 'abnormality': 0.7530
    },
    'BiomedCLIP': {
        'classification': 0.6090, 'density': 0.5642,
        'birads': 0.3268, 'abnormality': 0.6626
    },
}

task_labels = [TASK_DISPLAY[t] for t in TASKS]
N = len(TASKS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

model_colors = {'MammoSight': 'crimson', 'MedGemma-4B': 'steelblue',
                'MedGemma-27B': 'navy', 'MedImageInsight': 'forestgreen',
                'BiomedCLIP': 'darkorange'}
model_styles = {'MammoSight': '-', 'MedGemma-4B': '--',
                'MedGemma-27B': ':', 'MedImageInsight': '-.',
                'BiomedCLIP': '--'}

for model, scores in comparison_f1.items():
    vals = [scores[t] for t in TASKS] + [scores[list(TASKS.keys())[0]]]
    lw = 3 if model == 'MammoSight' else 1.5
    alpha = 0.25 if model == 'MammoSight' else 0
    ax.plot(angles, vals, model_styles[model],
            color=model_colors[model], linewidth=lw, label=model)
    ax.fill(angles, vals, color=model_colors[model], alpha=alpha)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(task_labels, size=11)
ax.set_ylim(0.2, 0.85)
ax.set_yticks([0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
ax.set_yticklabels(['0.3','0.4','0.5','0.6','0.7','0.8'], size=8)
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
ax.legend(loc='lower right', bbox_to_anchor=(1.35, -0.05), fontsize=9)
ax.set_title('Macro F1 Comparison — Radar Chart', y=1.08, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'radar_chart.pdf', bbox_inches='tight')
plt.show()

---
## 4. Segmentation Visualizations

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  4.1  Segmentation Baseline Comparison Bar Chart
# ────────────────────────────────────────────────────────────────────────────

seg_data = {
    'UNet3+'            : {'Dice': 0.021,  'IoU': 0.011,  'Type': 'Zero-shot'},
    'SAM 2.1 Hiera+'    : {'Dice': 0.023,  'IoU': 0.012,  'Type': 'Zero-shot'},
    'BiomedParse'       : {'Dice': 0.182,  'IoU': 0.134,  'Type': 'Zero-shot'},
    'SAM base'          : {'Dice': 0.572,  'IoU': 0.434,  'Type': 'Zero-shot'},
    'SAM-Med2D'         : {'Dice': 0.625,  'IoU': 0.485,  'Type': 'Zero-shot'},
    'SAM-Med2D FT'      : {'Dice': 0.734,  'IoU': 0.603,  'Type': 'Fine-tuned'},
    'MammoSight'        : {'Dice': 0.947,  'IoU': 0.950,  'Type': 'Fine-tuned'},
}
seg_df = pd.DataFrame(seg_data).T.reset_index()
seg_df.columns = ['Model', 'Dice', 'IoU', 'Type']
seg_df = seg_df.sort_values('Dice')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
type_colors = {'Zero-shot': '#7fcdbb', 'Fine-tuned': '#2c7fb8'}

for ax, metric in zip(axes, ['Dice', 'IoU']):
    colors = [type_colors[t] for t in seg_df['Type']]
    bars = ax.barh(seg_df['Model'], seg_df[metric], color=colors,
                   edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, seg_df[metric]):
        xpos = bar.get_width() + 0.005
        bold = val == seg_df[metric].max()
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10,
                fontweight='bold' if bold else 'normal')
    ax.set_xlabel(metric, fontsize=12)
    ax.set_xlim(0, 1.12)
    ax.set_title(f'{metric} Score Comparison', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axvline(x=seg_df.loc[seg_df.Model=='SAM-Med2D FT', metric].values[0],
               color='orange', linestyle='--', linewidth=1.5,
               label='Best prior FT method')
    ax.legend(fontsize=9)

# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in type_colors.items()]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Segmentation Performance — MammoBench Real Test Set (N=449)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'segmentation_comparison.pdf', bbox_inches='tight')
plt.show()

---
## 5. Attention & Saliency Visualizations (Plug-and-Play)

This section uses **pytorch-grad-cam** which provides the most comprehensive,
well-maintained CAM toolkit. We implement multiple methods so you can compare.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  5.1  Load Model
# ────────────────────────────────────────────────────────────────────────────

from configs.config_medimageinsight import Config
from models.model import MammoSightModel
from dataset.dataset import MammoBenchDataset
import torch.nn as nn
import math

cfg = Config()

model = MammoSightModel(
    backbone_type=cfg.BACKBONE,
    backbone_name=cfg.BACKBONE_NAME,
    sam_checkpoint_path=None,   # skip SAM for saliency (faster)
    pooling_mode=cfg.POOLING_MODE,
    fusion_mode=cfg.FUSION_MODE,
).to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt, strict=False)
model.eval()
print("Model loaded.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  5.2  Improved Single-View Wrapper for Saliency
#       Bypasses fusion — hooks directly at adapter output
# ────────────────────────────────────────────────────────────────────────────

class SaliencyWrapper(nn.Module):
    """
    Wraps MammoSight for single-view saliency computation.
    Bypasses cross-view fusion to avoid degenerate gradients from
    the zero-padded absent view.
    """
    def __init__(self, model, view='cc', task='classification'):
        super().__init__()
        self.model = model
        self.view  = view
        self.task  = task
        self.head  = getattr(model, f'head_severity'
                             if task == 'classification' else
                             f'head_{task}')

    def forward(self, x):
        sam_feats, global_pool = self.model.forward_view(x)

        if self.model.pooling_mode == 'sam_roi':
            # Use global pool doubled as proxy (no mask available)
            feat = torch.cat([global_pool, global_pool], dim=1)
        elif self.model.pooling_mode == 'learned':
            roi = self.model.saliency_head(sam_feats)
            roi_pool = (sam_feats * roi).sum(dim=(2,3)) / (
                roi.sum(dim=(2,3)) + 1e-6)
            feat = torch.cat([global_pool, roi_pool], dim=1)
        else:
            feat = global_pool

        B = x.shape[0]
        age_emb = self.model.age_encoder(
            torch.zeros(B, 1, device=x.device))
        combined = torch.cat([feat, age_emb], dim=1)
        return self.head(combined)


# def get_target_layers(model):
#     """Return appropriate target layers for MedImageInsight (DaViT)."""
#     bt = model.backbone_type
#     layers = []
#     if bt == 'medimageinsight':
#         bb = model.backbone
#         # DaViT: try last block's norm
#         if hasattr(bb, 'blocks') and len(bb.blocks) > 0:
#             last = bb.blocks[-1]
#             if hasattr(last, 'norm1'):
#                 layers = [last.norm1]
#             elif hasattr(last, 'blocks') and len(last.blocks) > 0:
#                 layers = [last.blocks[-1].norm1]
#         elif hasattr(bb, 'layers') and len(bb.layers) > 0:
#             layers = [bb.layers[-1].blocks[-1].norm1]
#     elif bt == 'swin':
#         bb = model.backbone
#         if hasattr(bb, 'layers'):
#             layers = [bb.layers[-1].blocks[-1].norm1]
#     # Fallback: adapter conv2 always works
#     if not layers:
#         layers = [model.adapter.conv2]
#     return layers

def get_target_layers(model):
    """Return appropriate target layers for GradCAM."""
    # adapter.conv2 is always present, spatial [B,256,64,64] — ideal for CAM
    return [model.adapter.conv2]

print("Saliency wrapper defined.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  5.3  Multi-Method Saliency Comparison Panel
#       Shows GradCAM++, EigenCAM, LayerCAM side by side
# ────────────────────────────────────────────────────────────────────────────

def load_and_preprocess_image(img_path, img_size=224):
    """Load and preprocess a mammogram image."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Cannot load: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img_float = img.astype(np.float32) / 255.0

    tensor = torch.from_numpy(
        ((img_float - mean) / std).transpose(2, 0, 1)
    ).unsqueeze(0).float()

    return img_float, tensor


def generate_multi_cam_panel(model, img_path, view='cc',
                             task='classification', target_class=None,
                             save_path=None):
    """
    Generate a publication-ready panel with multiple CAM methods.
    """
    if not HAS_GRADCAM:
        print("grad-cam not installed. pip install grad-cam")
        return

    img_float, tensor = load_and_preprocess_image(img_path)
    tensor = tensor.to(DEVICE)

    wrapper = SaliencyWrapper(model, view=view, task=task)
    target_layers = get_target_layers(model)

    # if target_class is None:
    #     with torch.no_grad():
    #         logits = wrapper(tensor)
    #         from utils.metrics import ordinal_logits_to_probs
    #         if task in ('classification', 'density', 'birads'):
    #             probs = ordinal_logits_to_probs(logits)
    #         else:
    #             probs = F.softmax(logits, dim=1)
    #         target_class = int(probs.argmax(1).item())

    # targets = [ClassifierOutputTarget(target_class)]
    targets = None
    cam_methods = [
        ('GradCAM++',  GradCAMPlusPlus),
        ('EigenCAM',   EigenCAM),
        ('LayerCAM',   LayerCAM),
    ]

    cams = []
    for name, CamClass in cam_methods:
        with CamClass(model=wrapper, target_layers=target_layers) as cam:
            grayscale = cam(input_tensor=tensor, targets=targets)
            cams.append((name, grayscale[0]))

    # Plot
    n_cols = len(cams) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

    axes[0].imshow(img_float, cmap='gray' if img_float.mean() < 0.3 else None)
    axes[0].set_title(f'Original ({view.upper()})', fontweight='bold')
    axes[0].axis('off')

    cls_name = TASKS.get(task, ['?'])
    # pred_label = cls_name[target_class] if target_class < len(cls_name) else str(target_class)
    pred_label = cls_name[target_class] if target_class is not None and target_class < len(cls_name) else "predicted"

    for ax, (name, cam_map) in zip(axes[1:], cams):
        overlay = show_cam_on_image(img_float, cam_map, use_rgb=True)
        ax.imshow(overlay)
        ax.set_title(f'{name}', fontweight='bold')
        ax.axis('off')
        # Colorbar
        sm = plt.cm.ScalarMappable(cmap='jet',
                                    norm=plt.Normalize(0, 1))
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f'Task: {TASK_DISPLAY.get(task, task)} | '
                 f'Predicted: {pred_label}',
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()
    return cams


# ── Example usage ─────────────────────────────────────────────────────────
# Replace with an actual image path from your dataset
example_cc_path = None  # e.g., "/data/gaurav.bhole/FullDataset/some_cc.png"

# Auto-find first CC image from predictions
if example_cc_path is None:
    master_df = pd.read_csv(MASTER_CSV)
    test_df = master_df[master_df['split'] == 'test'].dropna(subset=['cc_path'])
    # Pick a malignant case for interesting saliency
    mal_rows = test_df[test_df.get('classification', test_df.iloc[:,0]) == 'Malignant']
    if len(mal_rows) > 0:
        row = mal_rows.iloc[0]
    else:
        row = test_df.iloc[0]
    example_cc_path = os.path.join(IMAGE_ROOT, str(row['cc_path']))

if os.path.exists(str(example_cc_path)):
    print(f"Generating saliency for: {example_cc_path}")
    generate_multi_cam_panel(
        model, example_cc_path, view='cc',
        task='classification',
        save_path=FIG_DIR / 'saliency_panel_example.pdf'
    )
else:
    print(f"Image not found: {example_cc_path}")
    print("Set example_cc_path to a valid image path to generate saliency maps.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  5.4  Batch Saliency Grid — Multiple Cases, One Task
#       Shows correct vs incorrect predictions side by side
# ────────────────────────────────────────────────────────────────────────────

def batch_saliency_grid(model, master_csv, image_root, task='classification',
                        n_correct=4, n_incorrect=4, cam_method=GradCAMPlusPlus,
                        save_path=None):
    """
    Creates a grid showing:
    - Top row: correctly classified cases
    - Bottom row: incorrectly classified cases
    Each cell shows: original | saliency overlay
    """
    if not HAS_GRADCAM:
        print("grad-cam not installed."); return

    master_df = pd.read_csv(master_csv)
    test_df   = master_df[master_df['split'] == 'test'].copy()

    # Merge with predictions
    preds_df = pd.read_csv(MAMMOSIGHT_PRED_CSV)
    test_df  = test_df.merge(
        preds_df[['breast_id', f'{task}_pred', f'{task}_target', f'{task}_valid']],
        on='breast_id', how='inner'
    )
    test_df = test_df[test_df[f'{task}_valid'] == 1]

    correct_df   = test_df[test_df[f'{task}_pred'] == test_df[f'{task}_target']]
    incorrect_df = test_df[test_df[f'{task}_pred'] != test_df[f'{task}_target']]
    correct_df   = correct_df.dropna(subset=['cc_path']).head(n_correct)
    incorrect_df = incorrect_df.dropna(subset=['cc_path']).head(n_incorrect)

    wrapper       = SaliencyWrapper(model, view='cc', task=task)
    target_layers = get_target_layers(model)
    cls_names     = TASKS[task]
    n_cols        = max(n_correct, n_incorrect)

    fig = plt.figure(figsize=(4 * n_cols, 9))
    gs  = gridspec.GridSpec(4, n_cols, hspace=0.05, wspace=0.05)

    def process_row(df_subset, row_start, label):
        for col_i, (_, row) in enumerate(df_subset.iterrows()):
            img_path = os.path.join(image_root, str(row['cc_path']))
            if not os.path.exists(img_path):
                continue
            try:
                img_float, tensor = load_and_preprocess_image(img_path)
                tensor = tensor.to(DEVICE)
                tgt_cls = int(row[f'{task}_target'])
                pred_cls = int(row[f'{task}_pred'])

                with cam_method(model=wrapper, target_layers=target_layers) as cam:
                    # grayscale = cam(input_tensor=tensor,
                    #                targets=[ClassifierOutputTarget(pred_cls)])
                    grayscale = cam(input_tensor=tensor, targets=None)
                overlay = show_cam_on_image(img_float, grayscale[0], use_rgb=True)

                ax1 = fig.add_subplot(gs[row_start, col_i])
                ax1.imshow(img_float, cmap='gray' if img_float.mean()<0.3 else None)
                ax1.axis('off')
                if col_i == 0:
                    ax1.set_ylabel(label, fontsize=10, fontweight='bold')
                color = 'green' if label == 'Correct' else 'red'
                ax1.set_title(
                    f"GT:{cls_names[tgt_cls]}\nPred:{cls_names[pred_cls]}",
                    fontsize=7, color=color
                )
                for spine in ax1.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(2)

                ax2 = fig.add_subplot(gs[row_start+1, col_i])
                ax2.imshow(overlay)
                ax2.axis('off')
                for spine in ax2.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(2)
            except Exception as e:
                print(f"  Error processing {img_path}: {e}")

    process_row(correct_df,   row_start=0, label='Correct')
    process_row(incorrect_df, row_start=2, label='Incorrect')

    fig.text(0.5, 1.01, f'{TASK_DISPLAY[task]} — GradCAM++ Saliency Maps',
             ha='center', fontsize=13, fontweight='bold')
    fig.text(0.5, 0.53, '▲ Correct Predictions  |  ▼ Incorrect Predictions',
             ha='center', fontsize=10, color='gray')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()

# Run it
batch_saliency_grid(
    model, MASTER_CSV, IMAGE_ROOT,
    task='classification',
    n_correct=4, n_incorrect=4,
    cam_method=GradCAMPlusPlus if HAS_GRADCAM else None,
    save_path=FIG_DIR / 'saliency_grid_classification.pdf'
)

---
## 6. Feature Space Analysis (UMAP / t-SNE)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  6.1  Extract Combined Features from Frozen Model
# ────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def extract_features(model, csv_path, image_root, cfg,
                     split='test', max_samples=2000):
    """
    Extract combined_features [B, 768] from MammoSight for visualization.
    """
    ds = MammoBenchDataset(
        csv_path, split, image_root,
        img_size=cfg.IMAGE_SIZE, backbone_type=cfg.BACKBONE
    )
    loader = DataLoader(
        ds, batch_size=cfg.BATCH_SIZE,
        shuffle=False, num_workers=2
    )

    all_feats, all_labels, all_sources = [], [], []
    model.eval()

    for batch in tqdm(loader, desc='Extracting features'):
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(DEVICE)
            elif isinstance(v, dict):
                for sk, sv in v.items():
                    if isinstance(sv, torch.Tensor):
                        batch[k][sk] = sv.to(DEVICE)

        out = model(batch, return_features=True)
        feats = out['combined_features'].cpu().numpy()
        all_feats.append(feats)

        # Severity label
        valid = batch['validity']['classification'].cpu().numpy().astype(bool)
        lab   = batch['labels']['classification'].cpu().numpy()
        all_labels.append(np.where(valid, lab, -1))
        all_sources.extend(batch['source_dataset'])

        if sum(len(f) for f in all_feats) >= max_samples:
            break

    feats  = np.concatenate(all_feats)[:max_samples]
    labels = np.concatenate(all_labels)[:max_samples]
    sources = all_sources[:max_samples]

    valid_mask = labels >= 0
    return feats[valid_mask], labels[valid_mask], np.array(sources)[valid_mask]


print("Extracting features from test set (up to 2000 samples)...")
feats, feat_labels, feat_sources = extract_features(
    model, MASTER_CSV, IMAGE_ROOT, cfg,
    split='test', max_samples=2000
)
print(f"Extracted {len(feats)} samples, feature dim={feats.shape[1]}")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  6.2  UMAP + t-SNE Side-by-Side, Colored by Class and Source
# ────────────────────────────────────────────────────────────────────────────

print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000,
            random_state=42, n_jobs=-1)
tsne_emb = tsne.fit_transform(feats)

if HAS_UMAP:
    print("Running UMAP...")
    umap_model = umap.UMAP(n_components=2, n_neighbors=30,
                           min_dist=0.1, metric='cosine',
                           random_state=42)
    umap_emb = umap_model.fit_transform(feats)

severity_names  = ['Normal', 'Benign', 'Malignant']
severity_colors = ['#2ecc71', '#3498db', '#e74c3c']

n_sources = len(set(feat_sources))
src_palette = dict(zip(sorted(set(feat_sources)),
                       sns.color_palette('tab10', n_sources)))

n_plots = 4 if HAS_UMAP else 2
fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 6))

def scatter_2d(ax, emb, color_vals, color_map, title, alpha=0.5, s=12):
    for val, (label, color) in color_map.items():
        mask = color_vals == val
        if mask.sum() == 0: continue
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[color], label=f"{label} (n={mask.sum()})",
                   alpha=alpha, s=s, rasterized=True)
    ax.set_title(title, fontweight='bold')
    ax.legend(markerscale=2, fontsize=8, loc='best')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

sev_cmap = {k: (severity_names[k], severity_colors[k])
            for k in range(len(severity_names))}
src_cmap = {v: (v, src_palette[v]) for v in set(feat_sources)}

scatter_2d(axes[0], tsne_emb, feat_labels, sev_cmap,
           't-SNE — Colored by Cancer Severity')
scatter_2d(axes[1], tsne_emb, feat_sources, src_cmap,
           't-SNE — Colored by Source Dataset', alpha=0.4)

if HAS_UMAP:
    scatter_2d(axes[2], umap_emb, feat_labels, sev_cmap,
               'UMAP — Colored by Cancer Severity')
    scatter_2d(axes[3], umap_emb, feat_sources, src_cmap,
               'UMAP — Colored by Source Dataset', alpha=0.4)

plt.suptitle('MammoSight Feature Space Analysis\n'
             'Combined features [B, 768] from test set',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_space.pdf', bbox_inches='tight')
plt.show()

---
## 7. Dataset Distribution Analysis

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  7.1  Label Distribution Heatmap
# ────────────────────────────────────────────────────────────────────────────

master_df_full = pd.read_csv(MASTER_CSV)
test_df_full   = master_df_full[master_df_full['split'] == 'test']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

task_col_map = {
    'classification': 'classification',
    'density':        'density',
    'birads':         'birads',
    'abnormality':    'abnormality',
}

for ax, (task, col) in zip(axes, task_col_map.items()):
    if col not in test_df_full.columns:
        ax.set_visible(False); continue

    sources   = sorted(test_df_full['source_dataset'].dropna().unique())
    cls_labels = TASKS[task]

    pivot = pd.crosstab(
        test_df_full['source_dataset'],
        test_df_full[col]
    )
    # Normalize per row (source)
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    sns.heatmap(
        pivot_norm, annot=pivot.values, fmt='d',
        cmap='YlOrRd', ax=ax, linewidths=0.5,
        vmin=0, vmax=1, cbar_kws={'label': 'Proportion'},
        annot_kws={'size': 8}
    )
    ax.set_title(f'{TASK_DISPLAY[task]} Distribution per Source\n'
                 '(cell color=proportion, number=count)',
                 fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Source Dataset')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Test Set Label Distribution by Source Dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'label_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  7.2  Dataset Size Sunburst / Treemap
# ────────────────────────────────────────────────────────────────────────────

split_source = master_df_full.groupby(
    ['split', 'source_dataset']
).size().reset_index(name='count')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
split_order = ['train', 'val', 'test']
split_colors_map = {'train': '#3498db', 'val': '#2ecc71', 'test': '#e74c3c'}

for ax, split in zip(axes, split_order):
    subset = split_source[split_source['split'] == split].sort_values(
        'count', ascending=False)
    if subset.empty:
        ax.set_visible(False); continue

    colors = sns.color_palette('tab10', len(subset))
    wedges, texts, autotexts = ax.pie(
        subset['count'], labels=subset['source_dataset'],
        autopct='%1.1f%%', colors=colors,
        pctdistance=0.82, startangle=90,
        wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
    )
    for t in autotexts: t.set_fontsize(8)
    for t in texts:     t.set_fontsize(9)

    total = subset['count'].sum()
    ax.set_title(f'{split.capitalize()} Split\n(N={total:,})',
                 fontweight='bold', color=split_colors_map[split])

plt.suptitle('MammoBench Dataset Composition by Split',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'dataset_composition.pdf', bbox_inches='tight')
plt.show()

---
## 8. Error Analysis

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  8.1  Confidence vs Correctness Analysis
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, task in zip(axes, TASKS):
    d = mammo_preds[task]
    max_prob = d['probs'].max(axis=1)  # model confidence
    correct  = (d['preds'] == d['targs']).astype(int)

    # Bin by confidence
    bins = np.linspace(0, 1, 11)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_acc, bin_count, bin_se = [], [], []

    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (max_prob >= lo) & (max_prob < hi)
        if mask.sum() == 0:
            bin_acc.append(np.nan); bin_count.append(0)
            bin_se.append(0); continue
        acc_b = correct[mask].mean()
        se    = np.sqrt(acc_b * (1 - acc_b) / max(mask.sum(), 1))
        bin_acc.append(acc_b); bin_count.append(mask.sum())
        bin_se.append(se)

    bin_acc   = np.array(bin_acc)
    bin_count = np.array(bin_count)
    bin_se    = np.array(bin_se)

    # Bar for sample counts
    ax2 = ax.twinx()
    ax2.bar(bin_centers, bin_count, width=0.09, alpha=0.25,
            color='steelblue', label='Sample count')
    ax2.set_ylabel('Sample Count', color='steelblue', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='steelblue')

    valid = ~np.isnan(bin_acc)
    ax.errorbar(bin_centers[valid], bin_acc[valid],
                yerr=bin_se[valid],
                fmt='o-', color='crimson', linewidth=2,
                capsize=4, label='Accuracy', zorder=5)
    ax.plot([0,1],[0,1],'k--', alpha=0.4, linewidth=1, label='Perfect calibration')
    ax.set_xlabel('Max Predicted Probability')
    ax.set_ylabel('Accuracy', color='crimson')
    ax.tick_params(axis='y', labelcolor='crimson')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_title(TASK_DISPLAY[task], fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

plt.suptitle('Confidence vs Correctness (Calibration Analysis)\n'
             'Error bars = ±1 SE, bars = sample count per bin',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confidence_vs_correctness.pdf')
plt.show()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  8.2  Per-Source Performance Breakdown
# ────────────────────────────────────────────────────────────────────────────

task = 'classification'
d    = mammo_preds[task]
sources = sorted(set(d['source']))

source_metrics = []
for src in sources:
    mask = d['source'] == src
    if mask.sum() < 10: continue
    yt = d['targs'][mask]; yp = d['preds'][mask]
    f1_val = f1_score(yt, yp, average='macro', zero_division=0)
    acc    = accuracy_score(yt, yp)
    mcc    = matthews_corrcoef(yt, yp)
    try:
        auc_val = roc_auc_score(yt, d['probs'][mask],
                                multi_class='ovr', average='macro')
    except:
        auc_val = float('nan')
    source_metrics.append({
        'Dataset': src, 'N': int(mask.sum()),
        'Accuracy': acc, 'Macro F1': f1_val,
        'MCC': mcc, 'AUC': auc_val
    })

src_df = pd.DataFrame(source_metrics).sort_values('Macro F1', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['Macro F1', 'MCC', 'AUC']):
    colors = plt.cm.RdYlGn(
        (src_df[metric] - src_df[metric].min()) /
        (src_df[metric].max() - src_df[metric].min() + 1e-8)
    )
    bars = ax.barh(src_df['Dataset'], src_df[metric],
                   color=colors, edgecolor='black', linewidth=0.5)
    for bar, (_, row) in zip(bars, src_df.iterrows()):
        ax.text(bar.get_width() + 0.005,
                bar.get_y() + bar.get_height()/2,
                f"{row[metric]:.3f} (n={row['N']})",
                va='center', fontsize=9)
    ax.set_title(f'{TASK_DISPLAY[task]} — {metric} by Dataset',
                 fontweight='bold')
    ax.set_xlim(0, 1.22)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Per-Source Dataset Performance Breakdown',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'per_source_performance.pdf', bbox_inches='tight')
plt.show()

display(src_df.style.format({
    'Accuracy': '{:.4f}', 'Macro F1': '{:.4f}',
    'MCC': '{:.4f}', 'AUC': '{:.4f}'
}).background_gradient(subset=['Macro F1'], cmap='RdYlGn'))

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  8.3  Misclassification Pattern Analysis
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Severity: what does model confuse malignant with?
task = 'classification'
d = mammo_preds[task]
cls_names = TASKS[task]

# Error type matrix: for each GT class, what was it confused as?
error_data = {}
for gt_idx, gt_name in enumerate(cls_names):
    mask_gt = d['targs'] == gt_idx
    errors = d['preds'][mask_gt & (d['preds'] != d['targs'])]
    error_counts = {cls_names[k]: np.sum(errors == k)
                    for k in range(len(cls_names)) if k != gt_idx}
    error_data[gt_name] = error_counts

error_df = pd.DataFrame(error_data).fillna(0).astype(int)

sns.heatmap(error_df, annot=True, fmt='d', cmap='Reds',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Misclassification Pattern\n(Row=Confused As, Col=True Class)',
                  fontweight='bold')
axes[0].set_xlabel('True Class')
axes[0].set_ylabel('Predicted As (Incorrect)')

# Confidence distribution: correct vs incorrect
d = mammo_preds['classification']
max_prob = d['probs'].max(axis=1)
correct  = d['preds'] == d['targs']

axes[1].hist(max_prob[correct],  bins=30, alpha=0.6,
             color='green', label=f'Correct (n={correct.sum()})', density=True)
axes[1].hist(max_prob[~correct], bins=30, alpha=0.6,
             color='red', label=f'Incorrect (n={(~correct).sum()})', density=True)
axes[1].axvline(max_prob[correct].mean(), color='darkgreen',
                linestyle='--', linewidth=2,
                label=f'Correct mean={max_prob[correct].mean():.3f}')
axes[1].axvline(max_prob[~correct].mean(), color='darkred',
                linestyle='--', linewidth=2,
                label=f'Incorrect mean={max_prob[~correct].mean():.3f}')
axes[1].set_xlabel('Max Predicted Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Confidence Distribution\nCorrect vs Incorrect Predictions',
                  fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(alpha=0.3)

# KS test
ks_stat, ks_p = stats.ks_2samp(max_prob[correct], max_prob[~correct])
axes[1].text(0.05, 0.95, f'KS test: D={ks_stat:.3f}, p={ks_p:.1e}',
             transform=axes[1].transAxes, fontsize=9,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Error Analysis — Cancer Severity Classification',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_analysis.pdf', bbox_inches='tight')
plt.show()

---
## 9. Summary Metrics Table (Publication-Ready LaTeX)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  9.1  Full Metrics Table with CIs
# ────────────────────────────────────────────────────────────────────────────

print("=" * 100)
print(f"{'Task':<22} | {'Acc':>8} | {'Macro F1':>18} | {'MCC':>18} | {'Bal.Acc':>18} | {'N':>6}")
print("=" * 100)

latex_rows = []
for task in TASKS:
    d  = mammo_preds[task]
    ci = ci_results[task]
    N  = len(d['targs'])

    acc_s = f"{ci['acc'][0]:.4f} [{ci['acc'][1]:.4f},{ci['acc'][2]:.4f}]"
    f1_s  = f"{ci['f1'][0]:.4f}  [{ci['f1'][1]:.4f},{ci['f1'][2]:.4f}]"
    mcc_s = f"{ci['mcc'][0]:.4f}  [{ci['mcc'][1]:.4f},{ci['mcc'][2]:.4f}]"
    bal_s = f"{ci['bal'][0]:.4f}  [{ci['bal'][1]:.4f},{ci['bal'][2]:.4f}]"

    print(f"{TASK_DISPLAY[task]:<22} | {acc_s:>8} | {f1_s:>18} | {mcc_s:>18} | {bal_s:>18} | {N:>6}")

    # LaTeX row
    latex_rows.append(
        f"  {TASK_DISPLAY[task]} & {ci['acc'][0]:.3f} [{ci['acc'][1]:.3f},{ci['acc'][2]:.3f}] "
        f"& {ci['f1'][0]:.3f} [{ci['f1'][1]:.3f},{ci['f1'][2]:.3f}] "
        f"& {ci['mcc'][0]:.3f} [{ci['mcc'][1]:.3f},{ci['mcc'][2]:.3f}] "
        f"& {ci['bal'][0]:.3f} [{ci['bal'][1]:.3f},{ci['bal'][2]:.3f}] "
        f"& {N} \\\\"
    )

print("=" * 100)

# Generate LaTeX table
latex = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{MammoSight classification performance on MammoBench test set.",
    r"Values reported as point estimate [95\% bootstrap CI, 10,000 iterations].}",
    r"\label{tab:mammosight_results}",
    r"\begin{tabular}{lccccc}",
    r"\toprule",
    r"Task & Accuracy & Macro F1 & MCC & Balanced Acc. & N \\",
    r"\midrule",
] + latex_rows + [
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
]
with open(FIG_DIR / 'results_table.tex', 'w') as f:
    f.write('\n'.join(latex))
print(f"\nLaTeX table saved → {FIG_DIR}/results_table.tex")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  9.2  Final Summary: All Figures Saved
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("All figures saved to:", FIG_DIR.absolute())
print("="*60)

saved = sorted(FIG_DIR.glob('*'))
for f in saved:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45}  {size_kb:>7.1f} KB")

print("\n" + "="*60)
print("FIGURE GUIDE FOR PAPER")
print("="*60)
guide = {
    'confusion_matrices.pdf'      : 'Main Fig — model performance overview',
    'roc_curves.pdf'              : 'Main Fig or Supp — ROC per task',
    'radar_chart.pdf'             : 'Main Fig — model comparison snapshot',
    'segmentation_comparison.pdf' : 'Main Fig — Table 1 visual',
    'saliency_panel_example.pdf'  : 'Main Fig — interpretability',
    'saliency_grid_classification.pdf': 'Supp — correct/incorrect saliency',
    'feature_space.pdf'           : 'Supp — representation quality',
    'per_class_f1.pdf'            : 'Supp — per-class breakdown',
    'pr_curves.pdf'               : 'Supp — precision-recall',
    'calibration_curves.pdf'      : 'Supp — model calibration',
    'confidence_vs_correctness.pdf': 'Supp — calibration analysis',
    'error_analysis.pdf'          : 'Supp — misclassification patterns',
    'label_distribution.pdf'      : 'Supp — dataset statistics',
    'dataset_composition.pdf'     : 'Supp — dataset composition',
    'per_source_performance.pdf'  : 'Supp — generalizability',
    'significance_tests.csv'      : 'Supp Table — statistical tests',
    'results_table.tex'           : 'Main Table (copy into paper)',
}
for fname, usage in guide.items():
    exists = '✓' if (FIG_DIR / fname).exists() else '○'
    print(f"  {exists}  {fname:<45} → {usage}")